In [ ]:
import boto3


In [ ]:
client = boto3.client("sagemaker", region_name="ap-south-1")



In [ ]:
resp = client.list_pipelines(MaxResults=100)
print("Pipelines found:", len(resp.get("PipelineSummaries", [])))
for p in resp.get("PipelineSummaries", []):
    print(p["PipelineName"], p["PipelineArn"], p["CreatedTime"])

In [1]:
import os
import boto3
import sagemaker
from pipeline import get_pipeline   # adjust import path to where your file is

region = "ap-south-1"
default_bucket = "beatit-ai-data"   # or leave None to let session pick default
pipeline_name = "beatit-ai-churn-pipeline"  # choose desired name

# get role
sess = sagemaker.session.Session(boto_session=boto3.Session(region_name=region))
role = sagemaker.session.get_execution_role(sess)  # works if running in Studio/notebook with role
# OR explicitly set role_arn = "arn:aws:iam::<account>:role/YourSageMakerRole"

pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=default_bucket,
    pipeline_name=pipeline_name,
    base_job_prefix="beatit-ai-churn",
    processing_instance_type="ml.m5.large",
    training_instance_type="ml.m5.large",
)

# Register/create the pipeline in SageMaker (this creates the Pipeline resource)
pipeline.upsert(role_arn=role)
print("Pipeline upserted:", pipeline.name)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading analysis config to {s3_uri}.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python

Pipeline upserted: beatit-ai-churn-pipeline


In [ ]:
import sagemaker
session = sagemaker.Session()
session.default_bucket()


In [ ]:
!pip install --upgrade sagemaker

In [5]:
# Clarify baseline generation script
import boto3
import time
from sagemaker import Session
from sagemaker import get_execution_role
from sagemaker.clarify import DataConfig, BiasConfig
from sagemaker.clarify import SageMakerClarifyProcessor

# ------------ CONFIG - REPLACE THESE ----------------
REGION = "ap-south-1"  # e.g. ap-south-1
# If running in Studio you can use get_execution_role()
ROLE = get_execution_role()  # or "arn:aws:iam::123456789012:role/YourSageMakerRole"

# Input data (CSV)
TRAINING_CSV_S3 = "s3://beatit-ai-data/raw/train.csv"  # REPLACE_ME

# Output location where Clarify will write statistics & constraints
BASE_OUTPUT_S3 = "s3://beatit-ai-data/beatit-ai-churn/"  # REPLACE_ME

# Clarify config specifics
LABEL_INDEX = 0          # label column index in headerless CSV (0-based)
FACET_INDEX = 1          # sensitive feature column index (registered_via) in headerless CSV

# If you want the script to automatically copy generated files INTO the pipeline baseline folders,
# set these to the S3 prefixes your pipeline expects (from pipeline step details).
# Example:
# s3://<default_bucket>/<base_job_prefix>/<pipeline-exec-id>/databiascheckstep/
PIPELINE_EXPECTED_BIAS_BASE = f"{BASE_OUTPUT_S3}databiascheckstep/"  # REPLACE_ME
PIPELINE_EXPECTED_QUALITY_BASE = f"{BASE_OUTPUT_S3}dataqualitycheckstep/"  # REPLACE_ME
# ----------------------------------------------------

sagemaker_session = Session(boto3.session.Session(region_name=REGION))
boto_s3 = boto3.client("s3", region_name=REGION)

clarify_processor = SageMakerClarifyProcessor(
    role=ROLE,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    sagemaker_session=sagemaker_session,
)

data_config = DataConfig(
    s3_data_input_path=TRAINING_CSV_S3,
    s3_output_path=BASE_OUTPUT_S3,
    label=LABEL_INDEX,
    dataset_type="text/csv",
)

bias_config = BiasConfig(
    label_values_or_threshold=[1],  # positive class
    facet_name=[FACET_INDEX],       # sensitive feature index
)

print("Starting Clarify baseline generation job...")
clarify_processor.run_bias(
    data_config=data_config,
    data_bias_config=bias_config,
    wait=True,
    logs=True,
)

# After run completes, list objects under BASE_OUTPUT_S3 to find generated files
print("\nListing generated S3 objects under:", BASE_OUTPUT_S3)
parsed = BASE_OUTPUT_S3.replace("s3://", "").split("/", 1)
bucket = parsed[0]
prefix = parsed[1] if len(parsed) > 1 else ""

s3 = boto3.client("s3", region_name=REGION)
resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
for obj in resp.get("Contents", []):
    print(obj["Key"])

# Typically Clarify writes under:
# <BASE_OUTPUT_S3>/baseline/statistics/...
# <BASE_OUTPUT_S3>/baseline/constraints/...
# Let's find the exact filenames:
def find_files(prefix_subpath):
    objs = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/{prefix_subpath}")
    return [o["Key"] for o in objs.get("Contents", [])] if objs.get("Contents") else []

stats = find_files("baseline/statistics")
cons = find_files("baseline/constraints")

print("\nFound statistics files:", stats)
print("Found constraint files: ", cons)

# Optional: automatically copy the first-generated stats/constraints to pipeline expected locations
def s3_copy_object(src_bucket, src_key, dest_s3_uri):
    dest_parsed = dest_s3_uri.replace("s3://", "").split("/", 1)
    dest_bucket = dest_parsed[0]
    dest_prefix = dest_parsed[1] if len(dest_parsed) > 1 else ""
    file_name = src_key.split("/")[-1]
    dest_key = f"{dest_prefix.rstrip('/')}/{'statistics' if 'statistics' in src_key else 'constraints'}/{file_name}"
    copy_source = {"Bucket": src_bucket, "Key": src_key}
    print(f"Copying s3://{src_bucket}/{src_key} -> s3://{dest_bucket}/{dest_key}")
    s3.copy(copy_source, dest_bucket, dest_key)
    return f"s3://{dest_bucket}/{dest_key}"

if PIPELINE_EXPECTED_BIAS_BASE and stats and cons:
    # copy the first found stats & constraints for bias (if any)
    src_stat = stats[0]
    src_cons = cons[0]
    s3_stats_dest = s3_copy_object(bucket, src_stat, PIPELINE_EXPECTED_BIAS_BASE + "baseline")
    s3_cons_dest = s3_copy_object(bucket, src_cons, PIPELINE_EXPECTED_BIAS_BASE + "baseline")
    print("Copied bias baseline files to pipeline expected bias base:", s3_stats_dest, s3_cons_dest)
else:
    print("Not copying bias baselines: either pipeline destinations not set or no generated files found.")

# If you also generated data quality files separately (or same job produced both),
# you can copy them similarly to PIPELINE_EXPECTED_QUALITY_BASE.
print("Done.")


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


Starting Clarify baseline generation job...


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:55                                                                                   │
│                                                                                                  │
│    52 )                                                                                          │
│    53                                                                                            │
│    54 print("Starting Clarify baseline generation job...")                                       │
│ ❱  55 clarify_processor.run_bias(                                                                │
│    56 │   data_config=data_config,                                                               │
│    57 │   data_bias_config=bias_config,                                                          │
│    58 │   wait=True,                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: SageMakerClarifyProcessor.run_bias() got an unexpected keyword argument 'data_bias_config'

In [9]:
from sagemaker.clarify import SageMakerClarifyProcessor
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
REGION = "ap-south-1"
clarify_image_uri = sagemaker.image_uris.retrieve("clarify", REGION)

# Specify your desired S3 output location
custom_output_s3_uri = "s3://beatit-ai-data/beatit-ai-churn/clarify_results"

clarify_processor = SageMakerClarifyProcessor(
    role=role,
    # image_uri=clarify_image_uri,  <-- REMOVE THIS LINE
    instance_count=1,
    instance_type='ml.c5.xlarge',
    sagemaker_session=sagemaker_session
)



INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [11]:
# Define your DataConfig with the correct output path parameter
TRAINING_CSV_S3 = "s3://beatit-ai-data/raw/train.csv"  # REPLACE_ME

# Output location where Clarify will write statistics & constraints
BASE_OUTPUT_S3 = "s3://beatit-ai-data/beatit-ai-churn/"  # REPLACE_ME

# Clarify config specifics
LABEL_INDEX = 0          # label column index in headerless CSV (0-based)
FACET_INDEX = 1          # sensitive feature column index (registered_via) in headerless CSV

data_config = DataConfig(
    s3_data_input_path=TRAINING_CSV_S3,
    s3_output_path=custom_output_s3_uri,
    label=LABEL_INDEX,
    dataset_type="text/csv",
)

# 1. Run Bias Detection Job
clarify_processor.run_bias(
    data_config=data_config,
    bias_config=bias_config,)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:19                                                                                   │
│                                                                                                  │
│   16 )                                                                                           │
│   17                                                                                             │
│   18 # 1. Run Bias Detection Job                                                                 │
│ ❱ 19 clarify_processor.run_bias(                                                                 │
│   20 │   data_config=data_config,                                                                │
│   21 │   bias_config=bias_config,)                                                               │
│   22                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/clarify.py:2265 in run_bias                    │
│                                                                                                  │
│   2262 │   │   │   │     the Trial Component will be unassociated.                               │
│   2263 │   │   │   │   * ``'TrialComponentDisplayName'`` is used for display in Amazon SageMake  │
│   2264 │   │   """  # noqa E501  # pylint: disable=c0301                                         │
│ ❱ 2265 │   │   analysis_config = _AnalysisConfigGenerator.bias(                                  │
│   2266 │   │   │   data_config,                                                                  │
│   2267 │   │   │   bias_config,                                                                  │
│   2268 │   │   │   model_config,                                                                 │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/clarify.py:2641 in bias                        │
│                                                                                                  │
│   2638 │   │   │   pre_training_methods=pre_training_methods,                                    │
│   2639 │   │   │   post_training_methods=post_training_methods,                                  │
│   2640 │   │   )                                                                                 │
│ ❱ 2641 │   │   analysis_config = cls._add_predictor(                                             │
│   2642 │   │   │   analysis_config, model_config, model_predicted_label_config                   │
│   2643 │   │   )                                                                                 │
│   2644 │   │   return analysis_config                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/clarify.py:2670 in _add_predictor              │
│                                                                                                  │
│   2667 │   │   │   │   "predicted_label_dataset_uri" not in analysis_config                      │
│   2668 │   │   │   │   and "predicted_label" not in analysis_config                              │
│   2669 │   │   │   ):                                                                            │
│ ❱ 2670 │   │   │   │   raise ValueError(                                                         │
│   2671 │   │   │   │   │   "model_config must be provided when `predicted_label_dataset_uri` or  │
│   2672 │   │   │   │   │   "`predicted_label` are not provided in data_config."                  │
│   2673 │   │   │   │   )                                   

In [ ]:
# Define your other configs (Bias and SHAP)
bias_config = BiasConfig(
    label_values_or_headers=[headers[-1]],
    facet_values_or_headers={"col1": ["value_a"]},
    single_label_dataset=True
)

shap_config = SHAPConfig(
    baseline="mean_abs",
    num_samples=100
)
